# Coming from pymatgen

matverse is not a replacement for [pymatgen](https://pymatgen.org/) and does
not try to be. pymatgen is the crystallography, the file formats, the symmetry
and most of the analysis; matverse is a substrate that holds a *dataset* of
materials and the results computed on them, with pymatgen doing the work
underneath.

So the question for a pymatgen user is not "what do I have to learn instead" —
it is "what does putting my structures in an object buy me". This page answers
that with the operation pymatgen users write most often: transformations.

The short version:

```python
# pymatgen
structures = [PrimitiveCellTransformation().apply_transformation(s)
              for s in structures]

# matverse
mv.transform.apply(md, 'PrimitiveCellTransformation')
```

The list comprehension loses the originals, keeps no record, and gives you a
new list you now have to keep aligned with everything else by hand. The call
deposits a variant, keeps the input, and writes down what it did.

In [1]:
import matverse as mv
import numpy as np
import pandas as pd

mv.pl.set_style()

🔬 Starting plot initialization...
🧪 Calculators available: 6
    • emt — EMT (LGPL-2.1)
    • lj — Lennard-Jones (LGPL-2.1)
    • mace-mpa — mace-mpa (unstated)
    • mace-omat — mace-omat (unstated)
    • sevennet — sevennet (unstated)
    • chgnet — chgnet (unstated)


🖥️ NVIDIA CUDA GPUs: 1
    • [CUDA 0] NVIDIA H100 80GB HBM3 — 79.1 GB, compute 9.0

                   __
   ____ ___  ____ _/ /__   _____  _____________
  / __ `__ \/ __ `/ __/ | / / _ \/ ___/ ___/ _ \
 / / / / / / /_/ / /_ | |/ /  __/ /  (__  )  __/
/_/ /_/ /_/\__,_/\__/ |___/\___/_/  /____/\___/

🔖 Version: 0.1.40   🧮 Functions: 180   📚 Tutorials: https://matverse.readthedocs.io/
✅ set_style complete.



## Every transformation, by name

There are around forty-five `Transformation` classes in pymatgen. matverse does
not wrap forty-five of them — it wraps the *idea*, and looks the class up by
name at call time.

In [2]:
found = mv.transform.available()
len(found)

38

In [3]:
pd.DataFrame([
    {"name": name, "group": entry["group"], "signature": entry["signature"][:56]}
    for name, entry in list(found.items())[:12]
])

,name,group,signature
0,AutoOxiStateDecorationTransformation,standard,"(symm_tol=0.1, max_radius=4, max_permutations=..."
1,ChargedCellTransformation,standard,(charge=0)
2,ConventionalCellTransformation,standard,"(symprec: 'float' = 0.01, angle_tolerance=5, i..."
3,DeformStructureTransformation,standard,"(deformation=((1, 0, 0), (0, 1, 0), (0, 0, 1)))"
4,DiscretizeOccupanciesTransformation,standard,"(max_denominator=5, tol: 'float | None' = None..."
5,OrderDisorderedStructureTransformation,standard,"(algo: 'int' = 0, symmetrized_structures: 'boo..."
6,OxidationStateDecorationTransformation,standard,(oxidation_states)
7,OxidationStateRemovalTransformation,standard,()
8,PartialRemoveSpecieTransformation,standard,"(specie_to_remove, fraction_to_remove, algo=0)"
9,PerturbStructureTransformation,standard,"(distance: 'float' = 0.01, min_distance: 'floa..."


Read from pymatgen at call time rather than from a table here, so a
transformation added upstream is available the day it lands.

It searches, which matters when you half-remember the name:

In [4]:
list(mv.transform.available(search="supercell"))

['SupercellTransformation', 'CubicSupercellTransformation']

## Loading a dataset

Real oxides, so the oxidation-state machinery later has something to bite on.

In [5]:
md = mv.datasets.load("oxides")[:3].copy()
mv.pp.describe(md)

md.obs[["name", "formula", "spacegroup", "nsites"]]

,name,formula,spacegroup,nsites
0,SrTiO3,SrTiO3,Pm-3m,5
1,TiO2,TiO2,C2/m,12
2,VO2,VO2,P4_2/mnm,6


## Applying one

In [6]:
mv.transform.apply(md, "PrimitiveCellTransformation")
mv.transform.apply(md, "PerturbStructureTransformation", distance=0.05)

mv.variants(md)

['input', 'primitivecell', 'perturbstructure']

Three variants where a list comprehension would have left you with one list and
no way back. `input` is still there, and every later call names which one it
wants — which is the reason "what structure was this energy computed on" stays
answerable in matverse and is a matter of memory in a script.

Each application records whether it worked, **per row**:

In [7]:
md.obs[["name", "formula", "primitivecell_ok", "perturbstructure_ok"]]

,name,formula,primitivecell_ok,perturbstructure_ok
0,SrTiO3,SrTiO3,True,True
1,TiO2,TiO2,True,True
2,VO2,VO2,True,True


### A failure on one row is not a failure of the call

A dataset where a transformation applies to some materials and not others is
the normal case, not the exceptional one. Decorating with strontium oxidation
states works for SrTiO₃ and cannot work for TiO₂ or VO₂:

In [8]:
mv.transform.apply(md, "OxidationStateDecorationTransformation",
                   oxidation_states={"Sr": 2, "Ti": 4, "O": -2},
                   key_added="sr_decorated")

md.obs[["name", "formula", "sr_decorated_ok"]]

,name,formula,sr_decorated_ok
0,SrTiO3,SrTiO3,True
1,TiO2,TiO2,True
2,VO2,VO2,False


In [9]:
md.uns["transform"]["sr_decorated"]["errors"]

["2: ValueError: Oxidation states not specified for all elements, missing={'V'}"]

The rows that failed kept their original structure and are flagged `False`, so
a screen can filter on it. The alternative — raising on the whole dataset, or
silently returning a mixture you cannot tell apart — is worse in both
directions.

### The parameters are part of the record

In [10]:
md.uns["transform"]["perturbstructure"]

{'transformation': 'PerturbStructureTransformation',
 'params': {'distance': '0.05'},
 'source': 'input',
 'n_ok': 3,
 'n_failed': 0,
 'errors': []}

## One-to-many

Some transformations return several structures per input. `apply` takes the
first; `expand` keeps them all as rows, with `obs['parent']` pointing back —
the same derived-axis shape as `mv.pp.defects`, `mv.mag.orderings` and
`mv.disorder.orderings`.

In [11]:
from pymatgen.core import Lattice, Structure

alloy = mv.data.from_structures([
    Structure(Lattice.cubic(3.8), [{"Cu": 0.5, "Au": 0.5}] * 4,
              [[0, 0, 0], [0, .5, .5], [.5, 0, .5], [.5, .5, 0]]),
])
mv.pp.describe(alloy)

orderings = mv.transform.expand(
    alloy, "OrderDisorderedStructureTransformation", n=3, no_oxi_states=True)
mv.pp.describe(orderings)

orderings.obs[["parent", "variant_index", "formula", "nsites"]]

,parent,variant_index,formula,nsites
0,0,0,CuAu,4
1,0,1,CuAu,4
2,0,2,CuAu,4


```{note}
Where a namespace already wraps a specific one-to-many transformation, use it.
`mv.disorder.orderings` does this same work and records domain metadata the
generic path cannot know about — whether the Ewald ranking is meaningful, for
instance. `mv.transform.expand` is for the ones nothing wraps yet.
```

## Chaining

A sequence, applied in order, deposited as one variant.

In [12]:
mv.transform.chain(md, [
    ("PrimitiveCellTransformation", {}),
    ("PerturbStructureTransformation", {"distance": 0.03}),
], key_added="prepared")

md.obs[["name", "prepared_ok"]]

,name,prepared_ok
0,SrTiO3,True
1,TiO2,True
2,VO2,True


In [13]:
md.uns["transform"]["prepared"]["chain"]

[{'transformation': 'PrimitiveCellTransformation', 'params': {}},
 {'transformation': 'PerturbStructureTransformation',
  'params': {'distance': '0.03'}}]

One variant rather than one per step, because the intermediates are usually not
interesting and storing them all would fill the object. The full sequence is in
`uns` and in the provenance, so the result is still reproducible from the
record.

## Oxidation states: the missing prerequisite

This one is worth its own function because it is behind several of pymatgen's
more confusing error messages.

| what you ran | what it said |
|---|---|
| `OrderDisorderedStructureTransformation` | `Element has no attribute oxi_state!` |
| Ewald ranking | *(no error — every structure scores zero)* |
| `DopingTransformation` | `Valences cannot be assigned!` |

All three are asking for the same thing.

In [14]:
mv.transform.oxidation_states(md)

md.obs[["name", "formula", "oxidation_states_ok", "charge_balanced"]]

,name,formula,oxidation_states_ok,charge_balanced
0,SrTiO3,SrTiO3,True,True
1,TiO2,TiO2,True,True
2,VO2,VO2,True,True


In [15]:
structure = mv.structures(md, "oxidized")[0]
[(str(site.specie), site.specie.oxi_state) for site in structure][:5]

[('Sr2+', 2.0), ('Ti4+', 4.0), ('O2-', -2.0), ('O2-', -2.0), ('O2-', -2.0)]

Bond-valence analysis read the states off the bond lengths, and the cells came
out charge balanced — which is the check worth running, because an unbalanced
assignment is a wrong assignment.

Three routes, and which one you want depends on what you have:

In [16]:
mv.transform.oxidation_states(md, method="guess", key_added="guessed")
mv.transform.oxidation_states(md, method={"Sr": 2, "Ti": 4, "V": 4, "O": -2},
                              key_added="explicit")

mv.variants(md)

['input',
 'primitivecell',
 'perturbstructure',
 'sr_decorated',
 'prepared',
 'oxidized',
 'guessed',
 'explicit']

### It fails on metals, and says so per row

Bond-valence analysis reads oxidation states off bond lengths, and for a metal
the concept does not apply. A dataset mixing oxides with alloys is normal, so
that is recorded rather than raised:

In [17]:
metals = mv.datasets.metals(["Cu", "Al"])
mv.pp.describe(metals)
mv.transform.oxidation_states(metals)

metals.obs[["name", "oxidation_states_ok"]]

,name,oxidation_states_ok
0,Cu,False
1,Al,False


In [18]:
metals.uns["oxidation_states"]["note"]

'bond-valence analysis fails on metals, where an oxidation state is not a meaningful quantity; that is recorded per row rather than raised'

## What you get for it

Everything above is one pymatgen call per structure underneath. What the object
adds is not new physics — it is that the results stay attached:

```python
md = mv.data.from_cif('candidates/')          # pymatgen parses them
mv.transform.oxidation_states(md)             # pymatgen assigns them
mv.transform.apply(md, 'PrimitiveCellTransformation')
mv.calc.relax(md, level='emt', source='primitivecell')
mv.thermo.hull(md, level='emt', source='relaxed_emt')
mv.screen.filter(md, e_above_hull_emt__lt=0.05)

md.write_h5ad('screen.h5ad')                  # all of it, in one file
```

Six calls, one object, and at the end a file that carries the structures, every
variant, every energy, the level of theory each was computed at, and the record
of how they were produced. Written as a script over lists, the same pipeline is
six lists you keep aligned by index and a comment explaining what `structures2`
was.

## Cells that `supercell` cannot reach

`mv.pp.supercell` scales the axes by integers. `mv.transform.setting` does what
that cannot: **non-diagonal** transformations, axis permutations and origin
shifts.

In [19]:
box = mv.data.from_structures([Structure(Lattice.orthorhombic(3., 4., 5.),
                                         ["Cu"], [[0, 0, 0]])])
mv.pp.describe(box)

for spec, key in [("2a,b,c;0,0,0", "doubled"),
                  ("a+b,a-b,c;0,0,0", "bct"),
                  ("a,b,c;1/2,0,0", "shifted")]:
    mv.transform.setting(box, spec, key_added=key)

box.obs[[c for c in box.obs.columns if c.endswith("volume_ratio")]].round(3)

,doubled_volume_ratio,bct_volume_ratio,shifted_volume_ratio
0,2.0,2.0,1.0


`a+b, a-b, c` is the one to look at: it doubles the volume and gives axes of
3√2 and 4√2, and **no integer scaling of the axes reaches it**. That is the
transformation that turns a cubic cell into the body-centred tetragonal one.

The other standing use is settings. The same space group has several, and
"Pnma or Pbnm" is a permanent annoyance in perovskite work — the same structure
with the axes named differently, so a comparison across the two silently fails.

```{note}
The convention is the International Tables one: the string names the **new basis
in terms of the old**, and the lattice transforms as L′ = L·P. So `b,a,-c`
permutes which Cartesian direction each axis points along and leaves the three
lengths, as a set, unchanged — the cell is relabelled, not reshaped.

`obs['<key>_volume_ratio']` is there because that distinction is easy to get
backwards: a transformation meant to relabel that instead resized is visible
rather than assumed.
```

## A number you can check against a textbook

Most of what a screening library computes can only be checked against another
calculation. The Madelung energy is different — it has a closed form, and that
makes it the one place the machinery can be verified outright.

In [20]:
rocksalt = Structure.from_spacegroup("Fm-3m", Lattice.cubic(5.64),
                                     ["Na", "Cl"], [[0, 0, 0], [.5, .5, .5]])
cscl = Structure(Lattice.cubic(4.11), ["Cs", "Cl"],
                 [[0, 0, 0], [.5, .5, .5]])

ionic = mv.data.from_structures([rocksalt, cscl])
mv.pp.describe(ionic)
mv.transform.oxidation_states(ionic)
mv.prop.electrostatic(ionic, source="oxidized")

ionic.obs[["formula", "electrostatic_energy",
           "electrostatic_per_formula_unit"]].round(4)

,formula,electrostatic_energy,electrostatic_per_formula_unit
0,NaCl,-35.6941,-8.9235
1,CsCl,-7.1310,-7.1310


| | matverse | α·e²/4πε₀r |
|---|---|---|
| NaCl, α = 1.747565 | −8.9235 | −8.924 |
| CsCl, α = 1.762675 | −7.1310 | −7.131 |

Two structure types, two different Madelung constants, both exact. Getting one
right could be a fitted coincidence; getting both is not.

```{note}
It needs **oxidation states** — a point-charge sum with no charges is zero — and
a neutral structure returns NaN rather than 0, because a zero would read as "no
electrostatic contribution" rather than "nobody assigned any charges".

It is a point-charge model: no covalency, no polarisation, no short-range
repulsion. The right tool for ranking cation orderings on a fixed lattice, which
is what `mv.disorder.orderings` uses it for, and the wrong one for comparing
different chemistries.
```

## The record

In [21]:
for step in mv.provenance(md):
    print(step)

data.from_structures
pp.describe(source='input')
transform.apply(name='PrimitiveCellTransformation', source='input', key_added='primitivecell', n_ok=3)
transform.apply(name='PerturbStructureTransformation', source='input', key_added='perturbstructure', n_ok=3)
transform.apply(name='OxidationStateDecorationTransformation', source='input', key_added='sr_decorated', n_ok=2)
transform.chain(steps=['PrimitiveCellTransformation', 'PerturbStructureTransformation'], source='input', key_added='prepared')
transform.oxidation_states(method='bond valence analysis', source='input', key_added='oxidized')
transform.oxidation_states(method='composition guess', source='input', key_added='guessed')
transform.oxidation_states(method='explicit', source='input', key_added='explicit')


## What matverse does not do

An earlier version of this page listed three things, and two of them were wrong
— not architectural limits, just untested assumptions. They are worth recording
because the corrections are more informative than the original claims.

**"Molecules are out of scope by construction."** False. A `Molecule` has a
composition, so `X` and `var` build for water exactly as for a crystal. The only
thing that failed was a decoder assuming a lattice. See
[Molecules](molecules.ipynb) — `mv.mol` now does point groups, covalent bonds,
fragments and matching, and one object holds molecules and crystals together.

**"Visualisation — Crystal Toolkit and VESTA do that better."** They do, for
real inspection. That was a reason not to compete, not a reason to have
nothing: `mv.pl.structure` draws either kind of material, interactively when
py3Dmol is installed. It is the quick look you take twenty times a day.

**"Running anything."** This one was half right, and the half that was wrong is
now `mv.utils.submit`, which shells out to `sbatch` and records the job id on
the object. matverse still runs no DFT and is not a workflow manager — atomate2,
quacc and AiiDA do that, and a fourth would be a maintenance liability. What it
adds is the link back: *which job is computing this dataset* is answerable from
the data rather than from shell history.

What is genuinely left outside:

- **File format coverage.** `mv.data` has doors for the common ones and hands
  the rest to `pymatgen.io`, which reads far more than matverse should try to.
- **Being a workflow engine.** Retrying, chaining and monitoring belong to the
  tools built for it.

```{seealso}
[Getting data in and out](data_io.ipynb) is every door into and out of the
object. [Disorder](disorder.ipynb) and [Interfaces](interfaces.ipynb) are the
namespaces that wrap specific transformation families with their domain
knowledge intact.
```